## Setup

Install the project dependencies before running the notebook:

```bash
python -m pip install -r requirements.txt
python -m pip install evaluate matplotlib sentencepiece
```

The dataset is expected under `dataset/`, with `data_train/` and `data_test/`
image directories and tab-separated `data_train.txt` and `data_test.txt`
label files.

In [ ]:
import torch

print(f'CUDA available: {torch.cuda.is_available()}')


## Imports

In [ ]:
import glob
import os
from dataclasses import dataclass
from pathlib import Path

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from PIL import Image
from tqdm.notebook import tqdm
from torch.utils.data import Dataset
from transformers import (
    AutoImageProcessor,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    default_data_collator,
)

plt.rcParams['figure.figsize'] = (12, 9)


In [ ]:
def seed_everything(seed_value):
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Training and Dataset Configurations

In [ ]:
@dataclass(frozen=True)
class TrainingConfig:
    BATCH_SIZE: int = 8
    EPOCHS: int = 25
    LEARNING_RATE: float = 5e-5


@dataclass(frozen=True)
class DatasetConfig:
    DATA_ROOT: Path = Path(os.getenv('DATA_ROOT', 'dataset'))


@dataclass(frozen=True)
class ModelConfig:
    # MODEL_NAME: str = os.getenv('MODEL_NAME', 'microsoft/trocr-base-handwritten')
    MODEL_NAME: str = os.getenv('MODEL_NAME', 'microsoft/trocr-small-printed')
    OUTPUT_DIR: Path = Path(os.getenv('OUTPUT_DIR', 'ocrmodel-xlm-roberta'))


## The SCUT Dataset

The SCUT dataset that we are using contains a total of 7703 images. Out of these 6052 are training samples and 1651 are validation samples. Here is the directory structure of the `scut_data` directory after downloading and extracting it.

```
dataset/
├── data_train
├── data_test
├── data_train.txt
└── data_test.txt
```

In [ ]:
import matplotlib.pyplot as plt
import os

def visualize(dataset_path):
    plt.figure(figsize=(15, 4))
    for i in range(10):
        plt.subplot(2, 5, i+1)
        all_images = os.listdir(f"{dataset_path}/data_train")
        image = plt.imread(f"{dataset_path}/data_train/{all_images[i]}")
        plt.imshow(image)
        plt.axis('off')
        # Setting Khmer font
        plt.rcParams['font.family'] = 'Khmer OS'
        plt.title(all_images[i].split('.')[0], fontname='Khmer OS')
    plt.show()

visualize(DatasetConfig.DATA_ROOT)


## Prepare Dataset

In [ ]:
def load_labels(path: Path) -> pd.DataFrame:
    labels = pd.read_csv(path, sep='\t', header=None, names=['file_name', 'text'])
    return labels.dropna().reset_index(drop=True)


train_df = load_labels(DatasetConfig.DATA_ROOT / 'data_train.txt')
test_df = load_labels(DatasetConfig.DATA_ROOT / 'data_test.txt')


In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
class CustomOCRDataset(Dataset):
    def __init__(self, root_dir, df, processor, max_target_length=128):
        self.root_dir = Path(root_dir)
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = self.root_dir / self.df.loc[idx, 'file_name']
        text = str(self.df.loc[idx, 'text'])
        if not image_path.is_file():
            raise FileNotFoundError(f'Image not found: {image_path}')

        with Image.open(image_path) as image:
            pixel_values = self.processor(
                image.convert('RGB'), return_tensors='pt'
            ).pixel_values.squeeze(0)
        labels = self.processor.tokenizer(
            text, padding='max_length', max_length=self.max_target_length,
            truncation=True,
        ).input_ids
        labels = [
            token if token != self.processor.tokenizer.pad_token_id else -100
            for token in labels
        ]
        return {'pixel_values': pixel_values, 'labels': torch.tensor(labels)}


In [ ]:
print("Number of training examples:", len(train_dataset))
print("Number of validation examples:", len(valid_dataset))

In [ ]:
encoding = train_dataset[0]
for k,v in encoding.items():
    print(k, v.shape)

In [ ]:
image = Image.open(train_dataset.root_dir / train_df.loc[0, 'file_name']).convert('RGB')
plt.imshow(image)
plt.axis('off')


In [ ]:
labels = encoding['labels']
labels[labels == -100] = processor.tokenizer.pad_token_id
label_str = processor.decode(labels, skip_special_tokens=True)
print(label_str)

In [ ]:
# Check the tokenizer vocabulary size
vocab_size = processor.tokenizer.vocab_size
print(f"Tokenizer vocabulary size: {vocab_size}")

## Initialize the Model

In [ ]:
model = VisionEncoderDecoderModel.from_pretrained(ModelConfig.MODEL_NAME, )
model.to(device)
print(model)
# Total parameters and trainable parameters.
total_params = sum(p.numel() for p in model.parameters())
print(f"{total_params:,} total parameters.")
total_trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad)
print(f"{total_trainable_params:,} training parameters.")

In [ ]:
# Resize token embeddings if necessary
model.decoder.resize_token_embeddings(len(tokenizer))

In [ ]:
print(model.config)

## Model Configurations

In [ ]:
# Set special tokens used for creating the decoder_input_ids from the labels.
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
# Set Correct vocab size.
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id

# model.config.max_length = 64
# model.config.early_stopping = True
# model.config.no_repeat_ngram_size = 3
# model.config.length_penalty = 2.0
# model.config.num_beams = 4

We use the AdamW optimizer here with a weight decay of 0.0005.

In [ ]:
optimizer = optim.AdamW(
    model.parameters(), lr=TrainingConfig.LEARNING_RATE, weight_decay=0.0005
)

## Evaluation Metric

In [ ]:
cer_metric = evaluate.load('cer')

In [ ]:
def compute_cer(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    # Decode the predictions and labels
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)

    # Check for empty strings in predictions and labels
    empty_label_indices = [i for i, l in enumerate(label_str) if not l]

    if empty_label_indices:
        print(f"Empty labels found at indices: {empty_label_indices}")

    # Filter out empty labels and corresponding predictions
    filtered_pred_str = [p for i, p in enumerate(pred_str) if i not in empty_label_indices]
    filtered_label_str = [l for l in label_str if l]

    # Ensure that the lengths of predictions and labels match after filtering
    if len(filtered_pred_str) != len(filtered_label_str):
        raise ValueError("Mismatch between number of predictions and labels after removing empty strings.")

    # Compute CER
    cer = cer_metric.compute(predictions=filtered_pred_str, references=filtered_label_str)

    return {"cer": cer}


## Training and Validation Loops

In [ ]:
training_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    eval_strategy='epoch',
    per_device_train_batch_size=TrainingConfig.BATCH_SIZE,
    per_device_eval_batch_size=TrainingConfig.BATCH_SIZE,
    fp16=torch.cuda.is_available(),
    output_dir=str(ModelConfig.OUTPUT_DIR),
    logging_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=5,
    report_to='tensorboard',
    num_train_epochs=TrainingConfig.EPOCHS,
    learning_rate=TrainingConfig.LEARNING_RATE,
)


In [ ]:
# Initialize trainer.
trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=processor.tokenizer,
    args=training_args,
    compute_metrics=compute_cer,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=default_data_collator
)

## Train

In [ ]:
res = trainer.train()

## Inference

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
trained_model = model.to(device)
trained_model.eval()


In [ ]:
from torch.utils.data import DataLoader

test_dataloader = DataLoader(valid_dataset, batch_size=16)

In [ ]:
labels = batch["labels"]
labels[labels == -100] = processor.tokenizer.pad_token_id
label_str = processor.batch_decode(labels, skip_special_tokens=True)
label_str

In [ ]:
print('Running evaluation...')
trained_model.eval()
all_predictions = []
all_labels = []

for batch in tqdm(test_dataloader):
    pixel_values = batch['pixel_values'].to(device)
    with torch.no_grad():
        generated_ids = trained_model.generate(pixel_values, max_new_tokens=128)

    labels = batch['labels'].clone()
    labels[labels == -100] = processor.tokenizer.pad_token_id
    all_predictions.extend(
        processor.batch_decode(generated_ids, skip_special_tokens=True)
    )
    all_labels.extend(processor.batch_decode(labels, skip_special_tokens=True))

final_score = cer_metric.compute(
    predictions=all_predictions,
    references=all_labels,
)
print(f'Final CER score: {final_score:.4f}')


In [ ]:
def read_and_show(image_path):
    """
    :param image_path: String, path to the input image.

    Returns:
        image: PIL Image.
    """
    image = Image.open(image_path).convert('RGB')
    return image

In [ ]:
def ocr(image, processor, model):
    """
    :param image: PIL Image.
    :param processor: Huggingface OCR processor.
    :param model: Huggingface OCR model.

    Returns:
        generated_text: the OCR'd text string.
    """
    # We can directly perform OCR on cropped images.
    pixel_values = processor(image, return_tensors='pt').pixel_values.to(device)
    generated_ids = model.generate(pixel_values)
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return generated_text

In [ ]:
def eval_new_data(
    data_path=str(DatasetConfig.DATA_ROOT / 'data_test' / '*'),
    num_samples=50,
):
    image_paths = glob.glob(data_path)
    for i, image_path in tqdm(enumerate(image_paths), total=len(image_paths)):
        if i >= num_samples:
            break
        image = read_and_show(image_path)
        text = ocr(image, processor, trained_model)
        plt.figure(figsize=(7, 4))
        plt.imshow(image)
        plt.axis('off')
        plt.show()
        print('Generated text:', text)
